In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import json
from PIL import Image  
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torch.nn.functional as F
import torch.nn as nn
from fastai.learner import Learner
from monai.networks.nets import resnet18
from fastai.data.core import DataLoaders
from fastai.metrics import accuracy
from fastai.losses import CrossEntropyLossFlat
from fastai.callback.all import SaveModelCallback, EarlyStoppingCallback
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
%matplotlib inline

In [ ]:
# import pandas as pd

# # Load datasets
# df = pd.read_csv('Final_Datasets/long_axis_heart.csv')
# df_test = pd.read_csv('Final_Datasets/test_data_incidence.csv')
# controls = pd.read_csv('Final_Datasets/imbalanced_control_iid.csv')

# # Remove test cases and controls
# test_iids = set(df_test['IID'])

# # Get all cases (CAD = 1) from long_axis_heart.csv, excluding test data
# cases_train = df[(df['CAD'] == 1) & (~df['IID'].isin(test_iids))]

# # Get all controls (IID in controls.csv and CAD = 0), excluding test data
# controls_train = df[(df['IID'].isin(controls['IID'])) & (df['CAD'] == 0) & (~df['IID'].isin(test_iids))]

# # Combine the imbalanced dataset
# df_imbalanced_train = pd.concat([cases_train, controls_train]).reset_index(drop=True)

# # Save the imbalanced dataset
# df_imbalanced_train.to_csv('Final_Datasets/train_resnet_heart_imbalanced.csv', sep=',', index=False)

# # Report the dataset statistics
# cases_count = len(cases_train)
# controls_count = len(controls_train)
# print(f"Number of cases: {cases_count}")
# print(f"Number of controls: {controls_count}")
# print(f"Imbalance ratio (controls to cases): {controls_count / cases_count:.2f}")

In [ ]:
# # Load the CSV file
# input_file = 'Final_Datasets/train_resnet_heart_imbalanced.csv'
# df = pd.read_csv(input_file)

# # Define old and new directory paths
# old_path = '/scratch/07880/devansh/CAD/20208_long_axis/Processed/'
# new_path = '/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/20208_long_axis/Processed/'

# # Update the FilePath column
# df['FilePath'] = df['FilePath'].str.replace(old_path, new_path, regex=False)

# # Save the updated DataFrame back to the same file
# output_file = 'Final_Datasets/train_resnet_heart_imbalanced.csv'
# df.to_csv(output_file, index=False)

# print(f"File saved successfully to {output_file}")

In [ ]:
# import pandas as pd

# # Load the CSV file
# input_file = 'Final_Datasets/test_data_incidence.csv'
# df = pd.read_csv(input_file)

# # Define old and new directory paths
# old_path = '/scratch/07880/devansh/CAD/20208_long_axis/Processed/'
# new_path = '/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/20208_long_axis/Processed/'

# # Update the FilePath column
# if 'FilePath' in df.columns:  # Ensure the column exists
#     df['FilePath'] = df['FilePath'].str.replace(old_path, new_path, regex=False)
# else:
#     print("FilePath column not found in the DataFrame.")

# # Save the updated DataFrame back to the same file
# output_file = 'Final_Datasets/test_data_incidence.csv'
# df.to_csv(output_file, index=False)

# print(f"File saved successfully to {output_file}")

In [ ]:
class NpyDataset(Dataset):
    def __init__(self, dataframe, image_column_name, label_column_name, channel_dim=2):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name
        self.channel_dim = channel_dim
        self.transform = transforms.Compose([
            transforms.Lambda(lambda x: torch.stack([x, x, x], dim=0) if x.size(0) == 18 else x),  # Stack to create 3 channels
            transforms.Resize((112, 112))  # Resize to 112x112
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        npy_path = self.dataframe.loc[idx, self.image_column_name]
        label = self.dataframe.loc[idx, self.label_column_name]
        image = np.load(npy_path)[:, :, :, self.channel_dim]  # Accessing the specified channel dimension
        image = image[17:35, :, :]  # Selecting frames 17 to 34
        image = torch.tensor(image, dtype=torch.float32)
        image = self.transform(image)  # Apply transforms
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'label': label, 'path': npy_path}

class HeartDiseaseModel:
    def __init__(self, train_df_path, test_df_path, image_column_name, label_column_name, batch_size=32, model_name='best_model'):
        self.train_df_path = train_df_path
        self.test_df_path = test_df_path
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name
        self.batch_size = batch_size
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self._prepare_data()
        self._prepare_model()

    def _prepare_data(self):
        train_df = pd.read_csv(self.train_df_path)
        test_df = pd.read_csv(self.test_df_path)

        self.train_dataset = NpyDataset(train_df, self.image_column_name, self.label_column_name)
        self.test_dataset = NpyDataset(test_df, self.image_column_name, self.label_column_name)

        self.train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=8)
        self.test_loader = DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=8)

    def _prepare_model(self):
        self.model = models.video.r3d_18(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)
        self.model.to(self.device)

        self.learn = Learner(
            DataLoaders(self.train_loader, self.test_loader),
            self.model,
            loss_func=CrossEntropyLossFlat(),
            metrics=[accuracy],
            wd=1e-4,
            cbs=[
                SaveModelCallback(fname=self.model_name, monitor='train_loss'),
                EarlyStoppingCallback(monitor='train_loss', patience=10)
            ]
        ).to_fp16()

    def train(self, epochs=5, lr=1e-5):
        self.learn.fine_tune(epochs, base_lr=lr)

    def extract_embeddings(self, loader):
        self.model.eval()
        embeddings, labels, paths = [], [], []
        with torch.no_grad():
            for batch in loader:
                images = batch['image']
                label_batch = batch['label']
                path_batch = batch['path']
                images = images.to(self.device)
                
                # Pass through the model until the final pooling layer
                x = self.model.stem(images)
                x = self.model.layer1(x)
                x = self.model.layer2(x)
                x = self.model.layer3(x)
                x = self.model.layer4(x)
                x = self.model.avgpool(x)
                x = torch.flatten(x, 1)
                
                embeddings.append(x.cpu().numpy())
                labels.append(label_batch.cpu().numpy())
                paths.extend(path_batch)  # Collecting the paths
        embeddings = np.concatenate(embeddings)
        labels = np.concatenate(labels)
        return embeddings, labels, paths

    # Updated generate_embeddings_dataframe function
    def generate_embeddings_dataframe(self, embeddings, labels, paths):
        """
        Creates a Pandas DataFrame from embeddings, labels, and image paths.

        Args:
            embeddings (numpy.ndarray): The extracted embeddings.
            labels (numpy.ndarray): The labels corresponding to the embeddings.
            paths (list of str): The image paths.

        Returns:
            pd.DataFrame: A DataFrame with serialized embeddings and metadata.
        """
        # Serialize embeddings as JSON strings for safe CSV storage
        df = pd.DataFrame({
            'image_path': paths,
            'embedding': [json.dumps(emb.tolist()) for emb in embeddings],
            'label': labels
        })
        return df

    # Updated extract_and_save_embeddings function
    def extract_and_save_embeddings(self):
        """
        Extracts embeddings for train and test datasets and saves them as CSV files.

        The embeddings are serialized as JSON strings for robust CSV storage.
        """
        # Extract training embeddings
        train_embeddings, train_labels, train_paths = self.extract_embeddings(self.train_loader)
        train_df = self.generate_embeddings_dataframe(train_embeddings, train_labels, train_paths)

        # Extract test embeddings
        test_embeddings, test_labels, test_paths = self.extract_embeddings(self.test_loader)
        test_df = self.generate_embeddings_dataframe(test_embeddings, test_labels, test_paths)
        # Save DataFrames
        train_df.to_csv('train_embeddings_ch2_nov_imbalanced.csv', index=False)
        test_df.to_csv('test_embeddings_ch2_nov.csv', index=False)

In [ ]:
if __name__ == "__main__":
    model = HeartDiseaseModel(
        train_df_path='Final_Datasets/train_resnet_heart_imbalanced.csv',
        test_df_path='Final_Datasets/test_data_incidence.csv',
        image_column_name='FilePath',  # Replace with your image path column name
        label_column_name='CAD',       # Replace with your label column name
        model_name='heart_ch2_3channel_nov'
    )
    # Uncomment the following line if you need to train the model first
    # model.train(epochs=50, lr=1e-5)
    
    # Extract embeddings and save to CSV
    model.extract_and_save_embeddings()

In [ ]:
emb = pd.read_csv("train_embeddings_ch2_nov_imbalanced.csv")
emb['embedding'] = emb['embedding'].apply(lambda x: np.array(json.loads(x)))
emb.head()

In [ ]:
len(emb['embedding'][0])

**MONAI HEART**

In [ ]:
# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("heart_long_axis.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)


class HeartLongAxisDataset(Dataset):
    """
    Custom PyTorch Dataset for loading 3D heart long-axis images from .npy files.
    """
    def __init__(self, dataframe, image_column_name, label_column_name, custom_transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name

        # Default transformation pipeline
        default_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.Resize((112, 112))
        ])
        self.transform = custom_transform if custom_transform else default_transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        try:
            npy_path = self.dataframe.loc[idx, self.image_column_name]
            label = self.dataframe.loc[idx, self.label_column_name]
            
            # Load the 3D array for heart long-axis
            image = np.load(npy_path)[:, :, :, 0]  # Select the long-axis channel (0)
            image = image[17:33, :, :]  # Extract frames 17 to 32
            
            # Convert to tensor and add channel dimension
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)  # Shape: [1, depth, height, width]
            image = self.transform(image)  # Apply transformations
            
            return {'image': image, 'label': torch.tensor(label, dtype=torch.long), 'path': npy_path}
        
        except Exception as e:
            logger.error(f"Error loading image at index {idx}: {e}")
            raise


class HeartDiseaseEmbeddingExtractor:
    """
    Class for extracting embeddings from a fine-tuned MONAI ResNet18 model for heart long-axis images.
    """
    def __init__(self, train_df_path, test_df_path, image_column_name, label_column_name, batch_size=8, model_name='heart_long_axis_resnet18'):
        self.train_df_path = train_df_path
        self.test_df_path = test_df_path
        self.image_column_name = image_column_name
        self.label_column_name = label_column_name
        self.batch_size = batch_size
        self.model_name = model_name
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self._prepare_data()
        self._load_model()

    def _prepare_data(self):
        train_df = pd.read_csv(self.train_df_path)
        test_df = pd.read_csv(self.test_df_path)

        self.train_dataset = HeartLongAxisDataset(train_df, self.image_column_name, self.label_column_name)
        self.test_dataset = HeartLongAxisDataset(test_df, self.image_column_name, self.label_column_name)

        self.train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=8)
        self.test_loader = DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=8)

    def _load_model(self):
        # Initialize MONAI ResNet18
        self.model = resnet18(spatial_dims=3, n_input_channels=1, num_classes=2)
        
        # Use FastAI Learner to load the fine-tuned model
        self.learn = Learner(
            DataLoaders(self.train_loader, self.test_loader),
            self.model,
            loss_func=CrossEntropyLossFlat(),
            metrics=[accuracy]
        ).to_fp16()

        self.learn.load(self.model_name)
        self.model = self.learn.model.eval().to(self.device)

    def extract_embeddings(self, loader):
        embeddings, labels, paths = [], [], []
        with torch.no_grad():
            for batch in loader:
                images = batch['image'].to(self.device)
                label_batch = batch['label']
                path_batch = batch['path']

                # Forward pass to extract features
                x = self.model.conv1(images)  # Initial convolution
                x = self.model.bn1(x)        # BatchNorm
                x = self.model.maxpool(x)    # Max pooling
                x = self.model.layer1(x)
                x = self.model.layer2(x)
                x = self.model.layer3(x)
                x = self.model.layer4(x)
                x = self.model.avgpool(x)    # Global average pooling
                x = torch.flatten(x, 1)      # Flatten to get embeddings

                embeddings.append(x.cpu().numpy())
                labels.append(label_batch.cpu().numpy())
                paths.extend(path_batch)  # Collect paths
        
        embeddings = np.concatenate(embeddings)
        labels = np.concatenate(labels)
        return embeddings, labels, paths

     # Updated generate_embeddings_dataframe function
    def generate_embeddings_dataframe(self, embeddings, labels, paths):
        """
        Creates a Pandas DataFrame from embeddings, labels, and image paths.

        Args:
            embeddings (numpy.ndarray): The extracted embeddings.
            labels (numpy.ndarray): The labels corresponding to the embeddings.
            paths (list of str): The image paths.

        Returns:
            pd.DataFrame: A DataFrame with serialized embeddings and metadata.
        """
        # Serialize embeddings as JSON strings for safe CSV storage
        df = pd.DataFrame({
            'image_path': paths,
            'embedding': [json.dumps(emb.tolist()) for emb in embeddings],
            'label': labels
        })
        return df

    # Updated extract_and_save_embeddings function
    def extract_and_save_embeddings(self):
        """
        Extracts embeddings for train and test datasets and saves them as CSV files.

        The embeddings are serialized as JSON strings for robust CSV storage.
        """
        # Extract training embeddings
        train_embeddings, train_labels, train_paths = self.extract_embeddings(self.train_loader)
        train_df = self.generate_embeddings_dataframe(train_embeddings, train_labels, train_paths)

        # Extract test embeddings
        test_embeddings, test_labels, test_paths = self.extract_embeddings(self.test_loader)
        test_df = self.generate_embeddings_dataframe(test_embeddings, test_labels, test_paths)
        train_df.to_csv('train_embeddings_long_axis_ch0_imbalanced_monai.csv', index=False)
        test_df.to_csv('test_embeddings_long_axis_ch0_imbalanced_monai.csv', index=False)


In [ ]:
if __name__ == "__main__":
    extractor = HeartDiseaseEmbeddingExtractor(
        train_df_path='Final_Datasets/train_resnet_heart_imbalanced.csv',
        test_df_path='Final_Datasets/test_data_incidence.csv',
        image_column_name='FilePath',
        label_column_name='CAD',
        model_name='heart_ch0_3channel_MONAI_resnet18'
    )  
    
    # Extract and save embeddings
    extractor.extract_and_save_embeddings()

In [ ]:
emb = pd.read_csv("train_embeddings_long_axis_ch0_imbalanced_monai.csv")
emb['embedding'] = emb['embedding'].apply(lambda x: np.array(json.loads(x)))
emb.head()

In [ ]:
emb['embedding'][0]

In [ ]:
emb['label'].value_counts()

In [ ]:
em_te = pd.read_csv('test_embeddings_long_axis_ch2_imbalanced_monai.csv')
em_te.head()

In [ ]:
em_te['label'].value_counts()